# Practical 5: Sentiment Analysis Model

**Concept:** **Sentiment Analysis** is the task of determining whether a piece of text expresses a **positive**, **negative** (or neutral) opinion. It is a supervised text classification problem — we learn a mapping from text features to a sentiment label.

The classic pipeline is:
1. Preprocess the text (lowercase, remove noise)
2. Convert text to numeric features (**TF-IDF**)
3. Train a classifier on labeled examples
4. Evaluate with accuracy and a classification report

In this notebook we build and evaluate a sentiment analysis model using `scikit-learn` on a small, self-contained movie review dataset.

## Import Required Libraries

In [1]:
import re
import string

import nltk
from nltk.corpus import stopwords

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## Download NLTK Stopwords

In [2]:
nltk.download("stopwords")
nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package stopwords to /home/tejas/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /home/tejas/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/tejas/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## Create the Dataset
We create a small dataset of movie reviews labeled `1` (positive) and `0` (negative). It is large enough to train a working classifier but small enough to run instantly.

In [3]:
reviews = [
    "this movie is absolutely fantastic and exciting",
    "the plot was boring and the acting was terrible",
    "a wonderful story with brilliant performances",
    "i hated this film, it was a complete waste of time",
    "the visuals were stunning and the music was amazing",
    "worst movie i have ever seen, do not watch it",
    "an emotional and heartwarming masterpiece",
    "the dialogue felt fake and the ending was disappointing",
    "great action scenes and a smart script",
    "boring predictable and poorly directed",
    "i loved every minute of this beautiful movie",
    "the characters were dull and the movie dragged on",
    "an entertaining comedy with lots of laughs",
    "so sad, the story was confusing and badly told",
    "amazing cinematography and a gripping thriller",
    "the acting was mediocre and the plot made no sense",
    "a delightful feel-good movie, highly recommended",
    "slow, boring and full of clichés",
    "superb direction and outstanding cast",
    "a painful experience, i fell asleep twice"
]

labels = [1, 0, 1, 0, 1, 0, 1, 0, 1, 0,
          1, 0, 1, 0, 1, 0, 1, 0, 1, 0]

print("Dataset")
print("-" * 50)
for review, label in zip(reviews, labels):
    print(f"[{label}] {review}")
print(f"\nTotal samples: {len(reviews)}  |  Positive: {sum(labels)}  |  Negative: {len(labels) - sum(labels)}")

Dataset
--------------------------------------------------
[1] this movie is absolutely fantastic and exciting
[0] the plot was boring and the acting was terrible
[1] a wonderful story with brilliant performances
[0] i hated this film, it was a complete waste of time
[1] the visuals were stunning and the music was amazing
[0] worst movie i have ever seen, do not watch it
[1] an emotional and heartwarming masterpiece
[0] the dialogue felt fake and the ending was disappointing
[1] great action scenes and a smart script
[0] boring predictable and poorly directed
[1] i loved every minute of this beautiful movie
[0] the characters were dull and the movie dragged on
[1] an entertaining comedy with lots of laughs
[0] so sad, the story was confusing and badly told
[1] amazing cinematography and a gripping thriller
[0] the acting was mediocre and the plot made no sense
[1] a delightful feel-good movie, highly recommended
[0] slow, boring and full of clichés
[1] superb direction and outstanding 

## Text Preprocessing
We clean each review by lowercasing, removing punctuation and removing stopwords. This is the same pipeline introduced in Practicals 1 and 2.

In [4]:
# Load stopwords once
stop_words = set(stopwords.words("english"))

def clean_text(text):
    """Lowercase, remove punctuation and stopwords."""
    # Lowercase
    text = text.lower()

    # Remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))

    # Tokenize and remove stopwords
    tokens = [
        word for word in text.split()
        if word not in stop_words
    ]

    # Join back into a single string
    return " ".join(tokens)


# Preprocess all reviews
clean_reviews = [clean_text(review) for review in reviews]

print("Cleaned Reviews")
print("-" * 50)
for i, review in enumerate(clean_reviews):
    print(f"{i+1:>2}. {review}")

Cleaned Reviews
--------------------------------------------------
 1. movie absolutely fantastic exciting
 2. plot boring acting terrible
 3. wonderful story brilliant performances
 4. hated film complete waste time
 5. visuals stunning music amazing
 6. worst movie ever seen watch
 7. emotional heartwarming masterpiece
 8. dialogue felt fake ending disappointing
 9. great action scenes smart script
10. boring predictable poorly directed
11. loved every minute beautiful movie
12. characters dull movie dragged
13. entertaining comedy lots laughs
14. sad story confusing badly told
15. amazing cinematography gripping thriller
16. acting mediocre plot made sense
17. delightful feelgood movie highly recommended
18. slow boring full clichés
19. superb direction outstanding cast
20. painful experience fell asleep twice


## Convert Text to TF-IDF Features
Machines can't read words, so we convert each review into a TF-IDF vector (covered in Practical 3).

In [5]:
# Create TF-IDF features
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(clean_reviews)

print("TF-IDF Feature Matrix")
print("-" * 50)
print(f"Shape: {X.shape}  (documents, vocabulary size)")
print(f"Vocabulary size: {len(vectorizer.get_feature_names_out())}")

TF-IDF Feature Matrix
--------------------------------------------------
Shape: (20, 78)  (documents, vocabulary size)
Vocabulary size: 78


## Train / Test Split
We hold out 30% of the reviews for testing so we can honestly measure how well the model generalizes.

In [6]:
# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, labels,
    test_size=0.30,
    random_state=42,
    stratify=labels
)

print("Train / Test Split")
print("-" * 50)
print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples:  {X_test.shape[0]}")

Train / Test Split
--------------------------------------------------
Training samples: 14
Testing samples:  6


## Train the Classifier
**Logistic Regression** is a simple and effective baseline for text classification. It learns feature weights, so words like *"amazing"* push the prediction toward positive and words like *"boring"* toward negative.

In [7]:
# Train Logistic Regression
classifier = LogisticRegression(max_iter=1000, random_state=42)
classifier.fit(X_train, y_train)

# Predict on the test set
y_pred = classifier.predict(X_test)

print("Model Training Complete!")

Model Training Complete!


## Evaluate the Model
We measure **accuracy** and print a **classification report** (precision, recall, f1-score per class) plus a confusion matrix.

In [8]:
accuracy = accuracy_score(y_test, y_pred)

print("Model Evaluation")
print("-" * 50)
print(f"Test Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print()

print("Classification Report")
print("-" * 50)
print(classification_report(
    y_test, y_pred,
    target_names=["Negative", "Positive"]
))

print("Confusion Matrix")
print("-" * 50)
print(confusion_matrix(y_test, y_pred))

Model Evaluation
--------------------------------------------------
Test Accuracy: 0.6667 (66.67%)

Classification Report
--------------------------------------------------
              precision    recall  f1-score   support

    Negative       1.00      0.33      0.50         3
    Positive       0.60      1.00      0.75         3

    accuracy                           0.67         6
   macro avg       0.80      0.67      0.62         6
weighted avg       0.80      0.67      0.62         6

Confusion Matrix
--------------------------------------------------
[[1 2]
 [0 3]]


## Inspecting the Predictions
Let's compare the true labels and predicted labels for the test reviews.

In [9]:
print("True vs Predicted Labels")
print("-" * 50)

for i, (true_label, pred_label) in enumerate(zip(y_test, y_pred)):
    mark = "OK" if true_label == pred_label else "X"
    print(f"True: {true_label}  Predicted: {pred_label}  [{mark}]")

True vs Predicted Labels
--------------------------------------------------
True: 0  Predicted: 0  [OK]
True: 1  Predicted: 1  [OK]
True: 0  Predicted: 1  [X]
True: 1  Predicted: 1  [OK]
True: 1  Predicted: 1  [OK]
True: 0  Predicted: 1  [X]


## Predicting on New Reviews
Finally, we use the trained model to classify brand-new reviews it has never seen.

In [10]:
new_reviews = [
    "an incredible movie with a fantastic ending",
    "i wasted two hours of my life on this terrible film",
    "decent acting but a forgettable story"
]

# Preprocess and vectorize the new reviews
new_clean = [clean_text(review) for review in new_reviews]
new_vectors = vectorizer.transform(new_clean)

# Predict
new_predictions = classifier.predict(new_vectors)

print("Predictions on New Reviews")
print("-" * 50)

for review, pred in zip(new_reviews, new_predictions):
    sentiment = "Positive" if pred == 1 else "Negative"
    print(f"Review: {review}")
    print(f"Sentiment: {sentiment}")
    print()

Predictions on New Reviews
--------------------------------------------------
Review: an incredible movie with a fantastic ending
Sentiment: Positive

Review: i wasted two hours of my life on this terrible film
Sentiment: Negative

Review: decent acting but a forgettable story
Sentiment: Negative



## Bonus: Naive Bayes Comparison
**Multinomial Naive Bayes** is the classic baseline for text classification. We train it on the same features and compare accuracy with Logistic Regression.

In [11]:
# Train Multinomial Naive Bayes on the same features
nb_classifier = MultinomialNB()
nb_classifier.fit(X_train, y_train)
nb_pred = nb_classifier.predict(X_test)

nb_accuracy = accuracy_score(y_test, nb_pred)

print("Model Comparison")
print("-" * 50)
print(f"Logistic Regression Accuracy: {accuracy:.4f}")
print(f"Naive Bayes Accuracy:         {nb_accuracy:.4f}")

Model Comparison
--------------------------------------------------
Logistic Regression Accuracy: 0.6667
Naive Bayes Accuracy:         0.6667


## Summary

We built a complete sentiment analysis pipeline:
1. Created a labeled review dataset.
2. Preprocessed the text (lowercase, remove punctuation & stopwords).
3. Vectorized it with **TF-IDF**.
4. Trained **Logistic Regression** (and compared with **Naive Bayes**).
5. Evaluated with accuracy, classification report and confusion matrix.
6. Classified new unseen reviews.

This same TF-IDF + classifier recipe scales up to much larger datasets such as the 20 Newsgroups corpus used in Practical 7.